# importing libraries

In [2]:
!pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 2.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.1-py3-none-any.whl.metadata (10.0 kB)
Using cached pybind11-3.0.1-py3-none-any.whl (293 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4498213 sha256=fd14349894e90e0eddf76fb78c6b779782872c1d2e3dcd29df95d62f58818075
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


In [3]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
# from gensim.models import KeyedVectors
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, f1_score
# from gensim.models import FastText
import fasttext

# loading dataset

In [4]:
df = pd.read_csv("/content/spam.csv", encoding="latin-1")
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


# preprocessing

In [5]:
df = df.drop(["Unnamed: 2", "Unnamed: 3", "Unnamed: 4"], axis=1)
df.rename(columns={"v1": "label", "v2": "text"}, inplace=True)
df['label_enc'] = df['label'].map({'ham': 0, 'spam': 1})

X_train, X_test, y_train, y_test = train = train_test_split(df['text'], df['label_enc'], test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = X_train.to_numpy(), X_test.to_numpy(), y_train.to_numpy(), y_test.to_numpy()

y_train = to_categorical(y_train, num_classes=2)
y_test = to_categorical(y_test, num_classes=2)

# vocabulary stats

In [6]:
averageWordsLength = round(sum([len(i.split()) for i in df['text']]) / len(df['text']))
totalWordsLength = len(set(" ".join(df['text']).split()))
print(f"Data Loaded. Training samples: {len(X_train)}")
print(f"Average words per message: {averageWordsLength}")
print(f"Approximate vocabulary size: {totalWordsLength}")

Data Loaded. Training samples: 4457
Average words per message: 16
Approximate vocabulary size: 15686


# model development

In [7]:
cnn = tf.keras.models.Sequential()
cnn.add(tf.keras.layers.Input(shape=(1,), dtype=tf.string))

In [8]:
text_vec = tf.keras.layers.TextVectorization(
    max_tokens=totalWordsLength,
    standardize="lower_and_strip_punctuation",
    output_mode="int",
    output_sequence_length=averageWordsLength,
)
text_vec.adapt(X_train)
cnn.add(text_vec)

In [9]:
vocab_size = text_vec.get_vocabulary()
word_index = dict(zip(vocab_size, range(len(vocab_size))))

In [13]:
with open('training_data.txt', 'w') as f:
        for sentence in X_train:
            f.write(sentence.lower() + '\n')

In [16]:
sentences = [s.split() for s in X_train]
embedding_dim = 128

# Specify dim=embedding_dim to ensure the vectors match our matrix size
ft = fasttext.train_unsupervised('training_data.txt', epoch=1, lr=0.5, dim=embedding_dim)

embedding_matrix = np.zeros((len(vocab_size), embedding_dim))

for word, i in word_index.items():
    if word in ft:
        embedding_matrix[i] = ft[word]

In [17]:
cnn.add(tf.keras.layers.Embedding(
    input_dim=len(vocab_size),
    output_dim=embedding_dim,
    input_length=averageWordsLength,
    weights=[embedding_matrix],
    trainable=False
    )
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [18]:
cnn.add(tf.keras.layers.Conv1D(filters=128, kernel_size=5, activation="relu"))
cnn.add(tf.keras.layers.GlobalAveragePooling1D())
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))
cnn.add(tf.keras.layers.Dense(units=2, activation='softmax'))

In [19]:
cnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 16)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 16, 128)        │     1,089,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 12, 128)        │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,187,842 (4.53 MB)

 Trainable params: 98,818 (386.01 KB)

 Non-trainable params: 1,089,024 (4.15 MB)

In [20]:
cnn.compile(optimizer = 'adam', loss = 'categorical_crossentropy', metrics = ['accuracy'])

# model training

In [21]:
history = cnn.fit(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Epoch 1/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.8882 - loss: 0.2923 - val_accuracy: 0.9570 - val_loss: 0.1276
Epoch 2/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9654 - loss: 0.1074 - val_accuracy: 0.9614 - val_loss: 0.1063
Epoch 3/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9696 - loss: 0.1002 - val_accuracy: 0.9677 - val_loss: 0.0905
Epoch 4/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9740 - loss: 0.0765 - val_accuracy: 0.9722 - val_loss: 0.0793
Epoch 5/5
140/140 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.9859 - loss: 0.0513 - val_accuracy: 0.9713 - val_loss: 0.0769


# model evaluation

In [22]:
test_messages = [
    "Hey, are we meeting today?",
    "WIN cash now!!! Click the link",
    "Your OTP is 348921",
    "You have won $1000!",
    "Congratulations! You have won a free lottery ticket",
    "Don't forget to submit the assignment",
    "URGENT! You won a cash prize. Call immediately!",
    "My friend won a prize yesterday"
]

# Convert the list of messages to a TensorFlow constant of strings
test_messages_tf = tf.constant(test_messages, dtype=tf.string)

preds = cnn.predict(test_messages_tf)

for msg, p in zip(test_messages, preds):
    label = "Spam" if np.argmax(p)==1 else "Ham"
    print(f"{label} → {msg} ({p[0]:.2f})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step
Ham → Hey, are we meeting today? (1.00)
Ham → WIN cash now!!! Click the link (0.94)
Ham → Your OTP is 348921 (0.97)
Ham → You have won $1000! (0.97)
Ham → Congratulations! You have won a free lottery ticket (0.85)
Ham → Don't forget to submit the assignment (0.99)
Spam → URGENT! You won a cash prize. Call immediately! (0.01)
Ham → My friend won a prize yesterday (0.95)


In [23]:
print(classification_report(y_test.argmax(axis=1), cnn.predict(X_test).argmax(axis=1)))
print(f1_score(y_test.argmax(axis=1), cnn.predict(X_test).argmax(axis=1)))

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       966
           1       0.91      0.87      0.89       149

    accuracy                           0.97      1115
   macro avg       0.94      0.93      0.94      1115
weighted avg       0.97      0.97      0.97      1115

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
0.8904109589041096
